# PAHPA Baseline + Novelty 1 & 2: Confidence-Aware + Stability-Aware Scaling

**Base paper (PAHPA):** Point prediction with LightGBM + enhanced features → M/M/c queueing → scale pods

**Novelty 1:** Train quantile LightGBM (P10, P50, P90) to produce prediction intervals.
The interval width measures prediction confidence — the autoscaler scales conservatively
when uncertain and efficiently when confident.

**Novelty 2:** Stability filter prevents scaling flap by requiring consistent predictions
across multiple decision intervals before acting. Burst bypass ensures genuine spikes
are not delayed.

**Outputs to download from `output/`:**
- Models (point + quantile), predictions, metrics, plots, summary JSON

In [ ]:
!pip install -q lightgbm scikit-learn joblib pandas numpy matplotlib

# ============================================================
# Parse raw NASA HTTP log → minute-level QPS
# ============================================================
import re
from datetime import datetime
from collections import Counter

RAW_LOG = "/kaggle/input/datasets/sangabalanarsimha/nasa-http-july-1995-raw/access_log_Jul95"

import glob, os
candidates = glob.glob("/kaggle/input/*/access_log*") + glob.glob("/kaggle/input/*/*/access_log*")
if candidates:
    RAW_LOG = candidates[0]
    print(f"Found raw log: {RAW_LOG}")
else:
    print("WARNING: Could not find access_log. Listing /kaggle/input/:")
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            print(f"  {os.path.join(root, f)}")

timestamp_pattern = re.compile(r'\[(\d{2}/\w{3}/\d{4}:\d{2}:\d{2}:\d{2})')
minute_counts = Counter()
parse_errors = 0
total_lines = 0

with open(RAW_LOG, 'r', encoding='latin-1') as f:
    for line in f:
        total_lines += 1
        match = timestamp_pattern.search(line)
        if match:
            try:
                dt = datetime.strptime(match.group(1), '%d/%b/%Y:%H:%M:%S')
                minute_key = dt.replace(second=0)
                minute_counts[minute_key] += 1
            except ValueError:
                parse_errors += 1
        else:
            parse_errors += 1

print(f"Parsed {total_lines} lines, {parse_errors} errors")
print(f"Unique minutes: {len(minute_counts)}")

import pandas as pd
if minute_counts:
    min_time = min(minute_counts.keys())
    max_time = max(minute_counts.keys())
    all_minutes = pd.date_range(min_time, max_time, freq='min')
    qps_values = [minute_counts.get(t.to_pydatetime(), 0) for t in all_minutes]
    df_qps = pd.DataFrame({'timestamp': all_minutes, 'qps': qps_values})
    df_qps.to_csv("/kaggle/working/nasa_qps_history.csv", index=False)
    print(f"Saved {len(df_qps)} rows to /kaggle/working/nasa_qps_history.csv")
    print(f"QPS range: [{min(qps_values)}, {max(qps_values)}], mean: {sum(qps_values)/len(qps_values):.1f}")

## 1. Configuration

In [ ]:
import os
import json
import time
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from lightgbm import LGBMRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings("ignore", message="X does not have valid feature names")

DATASET_PATH = "/kaggle/working/nasa_qps_history.csv"
WINDOW_SIZE = 60
TEST_SIZE = 2880
SEED = 42
TARGET_TRANSFORM = "log1p"

# Quantile regression parameters (Novelty 1)
QUANTILES = [0.1, 0.5, 0.9]
CONFIDENCE_HIGH = 0.7
CONFIDENCE_LOW = 0.3

# Scaling simulation parameters
SERVICE_RATE = 30.0  # QPS per pod
SLO = 1.0            # seconds

OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Window: {WINDOW_SIZE}, Test size: {TEST_SIZE}, Transform: {TARGET_TRANSFORM}")
print(f"Quantiles: {QUANTILES}, Confidence thresholds: [{CONFIDENCE_LOW}, {CONFIDENCE_HIGH}]")

## 2. Load Data

In [ ]:
df = pd.read_csv(DATASET_PATH)
series_raw = df["qps"].astype(float).to_numpy()
print(f"Raw dataset: {len(series_raw)} data points")

TRIM_TO = min(36520, len(series_raw))
series = series_raw[:TRIM_TO]

print(f"Using first {len(series)} data points")
print(f"QPS range: [{series.min():.0f}, {series.max():.0f}], mean: {series.mean():.1f}")
print(f"Zero-QPS minutes: {np.sum(series == 0)} ({np.sum(series == 0)/len(series)*100:.1f}%)")

## 3. Feature Engineering

Enhanced features from QPS window: raw lags + first differences + multi-scale
rolling statistics (mean, std, min, max, median) + global stats (slope, range, ratios).
This gives LightGBM rich temporal context without needing LMD decomposition.

In [ ]:
def make_window_features(window_values):
    v = np.asarray(window_values, dtype=float).ravel()
    diffs = np.diff(v)
    mean_val = float(np.mean(v))
    std_val = float(np.std(v)) if len(v) > 1 else 0.0
    last = float(v[-1])

    features = list(v)
    features.extend(diffs.tolist())

    for size in (3, 6, 12, 24):
        seg = v[-min(size, len(v)):]
        features.extend([
            float(np.mean(seg)),
            float(np.std(seg)) if len(seg) > 1 else 0.0,
            float(np.min(seg)),
            float(np.max(seg)),
            float(np.median(seg)),
        ])

    slope = float(np.polyfit(np.arange(len(v), dtype=float), v, 1)[0]) if len(v) > 1 else 0.0
    features.extend([
        last, mean_val, std_val,
        float(np.min(v)), float(np.max(v)), float(np.ptp(v)),
        slope,
        float(diffs[-1]) if len(diffs) else 0.0,
        float(np.mean(diffs)) if len(diffs) else 0.0,
        float(np.std(diffs)) if len(diffs) > 1 else 0.0,
        last - mean_val,
        last / max(mean_val, 1e-9),
    ])
    return np.array(features, dtype=float)


def make_supervised(series_values, window):
    n = len(series_values)
    rows = n - window
    if rows <= 0:
        raise ValueError(f"Series too short: {n} <= {window}")
    sample = make_window_features(series_values[:window])
    feat_dim = len(sample)
    X = np.zeros((rows, feat_dim))
    Y = np.zeros(rows)
    for i in range(rows):
        X[i] = make_window_features(series_values[i:i + window])
        Y[i] = series_values[i + window]
    return X, Y


sample = make_window_features(np.random.randn(WINDOW_SIZE))
print(f"Feature dimension: {len(sample)} features per window of {WINDOW_SIZE}")

## 4. Model Helpers

In [ ]:
def build_lgbm(seed, objective="regression", alpha=None):
    params = dict(
        n_estimators=500,
        learning_rate=0.035,
        num_leaves=63,
        min_child_samples=12,
        reg_alpha=0.05,
        reg_lambda=0.15,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=seed,
        verbosity=-1,
        objective=objective,
    )
    if alpha is not None:
        params["alpha"] = alpha
    return LGBMRegressor(**params)


def transform_y(y):
    if TARGET_TRANSFORM == "log1p":
        return np.log1p(y)
    return y

def inverse_y(y):
    if TARGET_TRANSFORM == "log1p":
        return np.expm1(y)
    return y


def regression_metrics(y_true, y_pred, mape_floor=10.0):
    """Compute metrics. MAPE uses max(y_true, mape_floor) as denominator
    to avoid inflation from near-zero QPS values."""
    mask = y_true > 0
    denom = np.maximum(np.abs(y_true[mask]), mape_floor)
    mape = float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom) * 100)
    return {
        "mape": mape,
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)),
    }


def mmc_response_time(arrival_rate, service_rate, servers):
    if servers < 1 or service_rate <= 0:
        return math.inf
    if arrival_rate <= 0:
        return 1.0 / service_rate
    utilization = arrival_rate / (servers * service_rate)
    if utilization >= 1:
        return math.inf
    offered_load = arrival_rate / service_rate
    p0_terms = sum((offered_load**k) / math.factorial(k) for k in range(servers))
    tail = (offered_load**servers) / (math.factorial(servers) * (1 - utilization))
    p0 = 1.0 / (p0_terms + tail)
    erlang_c = tail * p0
    wait_queue = erlang_c / (servers * service_rate - arrival_rate)
    return wait_queue + (1.0 / service_rate)


def required_replicas(qps, service_rate=30.0, slo=1.0, min_r=1, max_r=15):
    for r in range(min_r, max_r + 1):
        if mmc_response_time(qps, service_rate, r) <= slo:
            return r
    return max_r


print("Model helpers + M/M/c queueing ready")

## 5. Build Dataset

In [ ]:
X, Y = make_supervised(series, WINDOW_SIZE)
split = len(X) - TEST_SIZE
split = min(max(split, 1), len(X) - 1)

X_train, X_test = X[:split], X[split:]
Y_train, Y_test = Y[:split], Y[split:]

print(f"Training: {len(X_train)} samples")
print(f"Test: {len(X_test)} samples")
print(f"Features: {X.shape[1]}")

## 6. PAHPA Baseline — Point Prediction (T+1)

In [ ]:
print("--- PAHPA Baseline (Point Prediction) ---")
t_start = time.time()

model_pahpa = build_lgbm(SEED)
model_pahpa.fit(X_train, transform_y(Y_train))
preds_pahpa = np.clip(inverse_y(model_pahpa.predict(X_test)), 0, None)

print(f"Done in {time.time()-t_start:.1f}s")

metrics_pahpa = regression_metrics(Y_test, preds_pahpa)
print(f"\n{'='*50}")
print(f"PAHPA Baseline T+1 Results:")
print(f"  MAPE: {metrics_pahpa['mape']:.2f}%  (paper: 24.51%)")
print(f"  MAE:  {metrics_pahpa['mae']:.4f}   (paper: 7.59)")
print(f"  RMSE: {metrics_pahpa['rmse']:.4f}  (paper: 10.53)")
print(f"  R²:   {metrics_pahpa['r2']:.4f}    (paper: 0.81)")
print(f"{'='*50}")

## 7. Novelty 1 — Quantile Regression (P10, P50, P90)

Train three LightGBM models with `objective='quantile'` to produce prediction intervals:
- **P10** (α=0.1) — optimistic lower bound
- **P50** (α=0.5) — median prediction
- **P90** (α=0.9) — conservative upper bound

The interval width (P90 − P10) measures prediction confidence.

In [ ]:
print("--- Novelty 1: Quantile Regression ---")
t_start = time.time()

quantile_models = {}
quantile_preds = {}

for q in QUANTILES:
    label = f"P{int(q*100)}"
    print(f"  Training {label} (alpha={q})...", end=" ")
    t0 = time.time()
    model = build_lgbm(SEED, objective="quantile", alpha=q)
    model.fit(X_train, transform_y(Y_train))
    preds = np.clip(inverse_y(model.predict(X_test)), 0, None)
    quantile_models[q] = model
    quantile_preds[q] = preds
    print(f"done ({time.time()-t0:.1f}s)")

print(f"Total quantile training: {time.time()-t_start:.1f}s")

p10 = quantile_preds[0.1]
p50 = quantile_preds[0.5]
p90 = quantile_preds[0.9]

## 8. Prediction Interval Analysis

In [ ]:
# P50 metrics — direct comparison with PAHPA point prediction
metrics_p50 = regression_metrics(Y_test, p50)

print(f"{'='*50}")
print(f"Quantile P50 (Median) Results:")
print(f"  MAPE: {metrics_p50['mape']:.2f}%")
print(f"  MAE:  {metrics_p50['mae']:.4f}")
print(f"  RMSE: {metrics_p50['rmse']:.4f}")
print(f"  R²:   {metrics_p50['r2']:.4f}")
print(f"{'='*50}")

# Prediction interval coverage and sharpness
coverage = np.mean((Y_test >= p10) & (Y_test <= p90)) * 100
avg_width = np.mean(p90 - p10)
median_width = np.median(p90 - p10)

print(f"\nPrediction Interval (P10-P90) Analysis:")
print(f"  Coverage: {coverage:.1f}% (target: ~80%)")
print(f"  Avg width: {avg_width:.2f} QPS")
print(f"  Median width: {median_width:.2f} QPS")

# Confidence scores
def compute_confidence(p10, p50, p90):
    width = p90 - p10
    return 1.0 - np.clip(width / np.maximum(p50, 1.0), 0, 2) / 2.0

confidence = compute_confidence(p10, p50, p90)

print(f"\nConfidence Score Distribution:")
print(f"  Mean: {np.mean(confidence):.3f}")
print(f"  High (>{CONFIDENCE_HIGH}):  {np.mean(confidence > CONFIDENCE_HIGH)*100:.1f}%")
print(f"  Medium:                     {np.mean((confidence > CONFIDENCE_LOW) & (confidence <= CONFIDENCE_HIGH))*100:.1f}%")
print(f"  Low (<={CONFIDENCE_LOW}):   {np.mean(confidence <= CONFIDENCE_LOW)*100:.1f}%")

## 9. Confidence-Aware QPS Target

The autoscaler blends between median (efficient) and upper bound (safe) based on confidence:
- **High confidence** → scale to P50 (median) — resource efficient
- **Medium confidence** → blend 50/50 between P50 and P90
- **Low confidence** → scale to P90 (upper bound) — avoids SLO violations

In [ ]:
def confidence_aware_qps(p10, p50, p90, high_thresh, low_thresh):
    confidence = compute_confidence(p10, p50, p90)
    target = np.where(
        confidence > high_thresh,
        p50,
        np.where(
            confidence > low_thresh,
            0.5 * p50 + 0.5 * p90,
            p90,
        )
    )
    return target, confidence


target_qps, confidence = confidence_aware_qps(p10, p50, p90, CONFIDENCE_HIGH, CONFIDENCE_LOW)

metrics_conf = regression_metrics(Y_test, target_qps)
print(f"Confidence-Aware QPS Target:")
print(f"  MAPE: {metrics_conf['mape']:.2f}%")
print(f"  MAE:  {metrics_conf['mae']:.4f}")

## 9b. Novelty 2 — Prediction-Stability-Aware Scaling Filter

Prevents scaling flap by requiring proposed changes to be consistent across
multiple consecutive decision intervals before acting:
- **Scale-up**: K consecutive intervals must all propose scaling up (default K=3)
- **Scale-down**: M out of N recent intervals must propose scaling down (default M=5, N=7)
- **Burst bypass**: Skip stability check when predicted load > 2× current capacity

In [ ]:
import collections

class StabilityFilter:
    def __init__(self, scale_up_consecutive=3, scale_down_m=5, scale_down_n=7,
                 burst_multiplier=2.0, service_rate_per_pod=30.0):
        self.scale_up_consecutive = scale_up_consecutive
        self.scale_down_m = scale_down_m
        self.scale_down_n = scale_down_n
        self.burst_multiplier = burst_multiplier
        self.service_rate_per_pod = service_rate_per_pod
        self._history = collections.deque(maxlen=scale_down_n)

    def filter(self, proposed_replicas, current_replicas, decision_qps=None):
        self._history.append(proposed_replicas)

        if proposed_replicas == current_replicas:
            return current_replicas

        direction = "up" if proposed_replicas > current_replicas else "down"

        # Burst bypass
        if (direction == "up" and decision_qps is not None
                and decision_qps > self.burst_multiplier * current_replicas * self.service_rate_per_pod):
            return proposed_replicas

        if direction == "up":
            if len(self._history) < self.scale_up_consecutive:
                return proposed_replicas
            recent = list(self._history)[-self.scale_up_consecutive:]
            if all(r > current_replicas for r in recent):
                return proposed_replicas
            return current_replicas

        # direction == "down"
        if len(self._history) < self.scale_down_n:
            return proposed_replicas
        recent_n = list(self._history)[-self.scale_down_n:]
        down_count = sum(1 for r in recent_n if r < current_replicas)
        if down_count >= self.scale_down_m:
            return proposed_replicas
        return current_replicas


SCALE_UP_K = 3
SCALE_DOWN_M = 5
SCALE_DOWN_N = 7
BURST_MULTIPLIER = 2.0

print(f"Stability filter: scale-up K={SCALE_UP_K}, scale-down M={SCALE_DOWN_M}/{SCALE_DOWN_N}, burst={BURST_MULTIPLIER}x")

## 10. Scaling Simulation — HPA vs PAHPA vs Confidence-Aware vs Stability-Aware

Simulate autoscaling decisions using M/M/c queueing theory across the test period.
Compare SLO violations, resource efficiency, and scaling stability across all four systems.

In [ ]:
n_test = len(Y_test)

hpa_replicas = np.zeros(n_test, dtype=int)
pahpa_replicas = np.zeros(n_test, dtype=int)
conf_replicas = np.zeros(n_test, dtype=int)
stab_replicas = np.zeros(n_test, dtype=int)

hpa_latency = np.zeros(n_test)
pahpa_latency = np.zeros(n_test)
conf_latency = np.zeros(n_test)
stab_latency = np.zeros(n_test)

stability_filter = StabilityFilter(
    scale_up_consecutive=SCALE_UP_K,
    scale_down_m=SCALE_DOWN_M,
    scale_down_n=SCALE_DOWN_N,
    burst_multiplier=BURST_MULTIPLIER,
    service_rate_per_pod=SERVICE_RATE,
)

for i in range(n_test):
    actual = Y_test[i]
    prev_actual = Y_test[i-1] if i > 0 else actual

    # HPA: reactive, uses previous minute's QPS
    hpa_r = required_replicas(prev_actual, SERVICE_RATE, SLO)
    hpa_replicas[i] = hpa_r
    hpa_latency[i] = mmc_response_time(actual, SERVICE_RATE, hpa_r)

    # PAHPA: uses point prediction
    pahpa_r = required_replicas(preds_pahpa[i], SERVICE_RATE, SLO)
    pahpa_replicas[i] = pahpa_r
    pahpa_latency[i] = mmc_response_time(actual, SERVICE_RATE, pahpa_r)

    # Confidence-aware: uses risk-adjusted target
    conf_r = required_replicas(target_qps[i], SERVICE_RATE, SLO)
    conf_replicas[i] = conf_r
    conf_latency[i] = mmc_response_time(actual, SERVICE_RATE, conf_r)

    # Stability-aware: confidence-aware + stability filter (Novelty 2)
    current_stab = stab_replicas[i-1] if i > 0 else 1
    stab_r = stability_filter.filter(
        proposed_replicas=conf_r,
        current_replicas=current_stab,
        decision_qps=target_qps[i],
    )
    stab_replicas[i] = stab_r
    stab_latency[i] = mmc_response_time(actual, SERVICE_RATE, stab_r)


def scaling_metrics(replicas, latency, actual_qps, slo=1.0):
    finite_lat = latency[np.isfinite(latency)]
    slo_violations = np.sum(latency > slo) / len(latency) * 100
    p99_latency = np.percentile(finite_lat, 99) if len(finite_lat) > 0 else float('inf')
    avg_pods = np.mean(replicas)
    changes = int(np.sum(np.diff(replicas) != 0))
    flaps = 0
    for j in range(2, len(replicas)):
        if replicas[j] > replicas[j-1] < replicas[j-2]:
            flaps += 1
    min_needed = np.array([required_replicas(q, SERVICE_RATE, slo) for q in actual_qps])
    over_prov = np.mean(np.maximum(replicas - min_needed, 0)) / max(np.mean(min_needed), 1) * 100
    return {
        "slo_violation_pct": round(slo_violations, 2),
        "p99_latency_ms": round(p99_latency * 1000, 1),
        "avg_pods": round(avg_pods, 2),
        "scaling_events": changes,
        "flap_count": flaps,
        "over_prov_pct": round(over_prov, 2),
    }


metrics_hpa_scale = scaling_metrics(hpa_replicas, hpa_latency, Y_test)
metrics_pahpa_scale = scaling_metrics(pahpa_replicas, pahpa_latency, Y_test)
metrics_conf_scale = scaling_metrics(conf_replicas, conf_latency, Y_test)
metrics_stab_scale = scaling_metrics(stab_replicas, stab_latency, Y_test)

print(f"\n{'='*110}")
print("Scaling Simulation Results (M/M/c queueing, SLO=1s)")
print(f"{'='*110}")
scale_df = pd.DataFrame([
    {"System": "HPA Baseline (reactive)", **metrics_hpa_scale},
    {"System": "PAHPA (point prediction)", **metrics_pahpa_scale},
    {"System": "Novelty 1 (confidence-aware)", **metrics_conf_scale},
    {"System": "Novelty 1+2 (conf + stability)", **metrics_stab_scale},
])
print(scale_df.to_string(index=False))
print(f"{'='*110}")

print(f"\nNovelty 2 improvements over Novelty 1 alone:")
print(f"  Scaling events: {metrics_conf_scale['scaling_events']} -> {metrics_stab_scale['scaling_events']} "
      f"({(1 - metrics_stab_scale['scaling_events']/max(metrics_conf_scale['scaling_events'],1))*100:.0f}% fewer)")
print(f"  Flap count: {metrics_conf_scale['flap_count']} -> {metrics_stab_scale['flap_count']} "
      f"({(1 - metrics_stab_scale['flap_count']/max(metrics_conf_scale['flap_count'],1))*100:.0f}% fewer)")
print(f"  SLO violations: {metrics_conf_scale['slo_violation_pct']}% -> {metrics_stab_scale['slo_violation_pct']}%")
print(f"  Avg pods: {metrics_conf_scale['avg_pods']} -> {metrics_stab_scale['avg_pods']}")

## 11. Comparison Tables

In [ ]:
print("\n" + "=" * 80)
print("Prediction Accuracy Comparison")
print("=" * 80)
pred_comparison = pd.DataFrame([
    {"System": "Paper SLMD-LightGBM", "mape": 24.51, "mae": 7.59, "rmse": 10.53, "r2": 0.81},
    {"System": "Our PAHPA (point)", **metrics_pahpa},
    {"System": "Novelty1 P50 (median)", **metrics_p50},
    {"System": "Novelty1 Conf-Aware target", **metrics_conf},
])
print(pred_comparison.to_string(index=False, float_format="{:.4f}".format))
print("=" * 80)

print(f"\nKey Novelty 1 Metrics:")
print(f"  Prediction interval coverage (P10-P90): {coverage:.1f}%")
print(f"  Average interval width: {avg_width:.2f} QPS")
print(f"  Mean confidence score: {np.mean(confidence):.3f}")

## 12. Visualizations

In [ ]:
# Plot 1: Actual vs Predicted with Prediction Intervals
plot_range = slice(0, min(500, n_test))
x_axis = np.arange(len(Y_test[plot_range]))

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Panel 1: PAHPA point prediction
axes[0].plot(x_axis, Y_test[plot_range], label="Actual", linewidth=0.8, color='black')
axes[0].plot(x_axis, preds_pahpa[plot_range], label="PAHPA Predicted", linewidth=0.8, alpha=0.8, color='#2196F3')
axes[0].set_title(f"PAHPA Point Prediction (MAPE: {metrics_pahpa['mape']:.2f}%)")
axes[0].legend()
axes[0].set_ylabel("QPS")

# Panel 2: Quantile predictions with intervals
axes[1].plot(x_axis, Y_test[plot_range], label="Actual", linewidth=0.8, color='black')
axes[1].plot(x_axis, p50[plot_range], label="P50 (median)", linewidth=0.8, color='#4CAF50')
axes[1].fill_between(x_axis, p10[plot_range], p90[plot_range], alpha=0.25, color='#4CAF50', label="P10-P90 interval")
axes[1].set_title(f"Quantile Prediction with 80% Interval (Coverage: {coverage:.1f}%)")
axes[1].legend()
axes[1].set_ylabel("QPS")

# Panel 3: Confidence-aware target
axes[2].plot(x_axis, Y_test[plot_range], label="Actual", linewidth=0.8, color='black')
axes[2].plot(x_axis, target_qps[plot_range], label="Conf-Aware Target", linewidth=0.8, color='#FF5722')
axes[2].plot(x_axis, p50[plot_range], label="P50", linewidth=0.5, alpha=0.5, color='#4CAF50', linestyle='--')
axes[2].plot(x_axis, p90[plot_range], label="P90", linewidth=0.5, alpha=0.5, color='#FFC107', linestyle='--')
axes[2].set_title(f"Confidence-Aware QPS Target (MAPE: {metrics_conf['mape']:.2f}%)")
axes[2].legend()
axes[2].set_ylabel("QPS")
axes[2].set_xlabel("Test sample index")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/actual_vs_predicted.png", dpi=150)
plt.show()

In [ ]:
# Plot 2: Confidence distribution and its effect on scaling
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Confidence histogram
axes[0, 0].hist(confidence, bins=50, color='#2196F3', edgecolor='white', alpha=0.8)
axes[0, 0].axvline(CONFIDENCE_HIGH, color='green', linestyle='--', label=f'High thresh ({CONFIDENCE_HIGH})')
axes[0, 0].axvline(CONFIDENCE_LOW, color='red', linestyle='--', label=f'Low thresh ({CONFIDENCE_LOW})')
axes[0, 0].set_title('Confidence Score Distribution')
axes[0, 0].set_xlabel('Confidence')
axes[0, 0].set_ylabel('Count')
axes[0, 0].legend()

# Interval width vs actual QPS
interval_width = p90 - p10
axes[0, 1].scatter(Y_test, interval_width, s=1, alpha=0.3, color='#FF5722')
axes[0, 1].set_title('Prediction Interval Width vs Actual QPS')
axes[0, 1].set_xlabel('Actual QPS')
axes[0, 1].set_ylabel('P90 - P10 (interval width)')

# Scaling comparison over time (all 4 systems)
scale_range = slice(0, min(500, n_test))
axes[1, 0].plot(hpa_replicas[scale_range], label='HPA', linewidth=0.8, alpha=0.6)
axes[1, 0].plot(pahpa_replicas[scale_range], label='PAHPA', linewidth=0.8, alpha=0.6)
axes[1, 0].plot(conf_replicas[scale_range], label='Conf-Aware', linewidth=0.8, alpha=0.7, color='#FF5722')
axes[1, 0].plot(stab_replicas[scale_range], label='Conf+Stability', linewidth=1.2, alpha=0.9, color='#9C27B0')
axes[1, 0].set_title('Replica Count Over Time')
axes[1, 0].set_xlabel('Time step')
axes[1, 0].set_ylabel('Replicas')
axes[1, 0].legend()

# Latency comparison (CDF) — all 4 systems
for label, lat, color in [
    ('HPA', hpa_latency, '#2196F3'),
    ('PAHPA', pahpa_latency, '#4CAF50'),
    ('Conf-Aware', conf_latency, '#FF5722'),
    ('Conf+Stability', stab_latency, '#9C27B0'),
]:
    finite = np.sort(lat[np.isfinite(lat)])
    cdf = np.arange(1, len(finite)+1) / len(finite)
    axes[1, 1].plot(finite * 1000, cdf, label=label, linewidth=1.5, color=color)
axes[1, 1].axvline(SLO * 1000, color='red', linestyle=':', label=f'SLO ({SLO*1000:.0f}ms)')
axes[1, 1].set_title('Latency CDF')
axes[1, 1].set_xlabel('Response Time (ms)')
axes[1, 1].set_ylabel('Cumulative Probability')
axes[1, 1].set_xlim(0, min(SLO * 3000, 5000))
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/scaling_analysis.png", dpi=150)
plt.show()

In [ ]:
# Plot 3: QPS overview
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(series, linewidth=0.3, color='#2196F3')
ax.axvline(len(series) - TEST_SIZE, color='red', linestyle='--', label=f'Train/Test split')
ax.set_title(f'NASA HTTP QPS — Full Series ({len(series)} minutes)')
ax.set_xlabel('Minute')
ax.set_ylabel('QPS')
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/qps_overview.png", dpi=150)
plt.show()

## 13. Save All Outputs

In [ ]:
# Save predictions
preds_df = pd.DataFrame({
    "sample_index": np.arange(n_test),
    "actual": Y_test,
    "pahpa_point": preds_pahpa,
    "quantile_p10": p10,
    "quantile_p50": p50,
    "quantile_p90": p90,
    "confidence": confidence,
    "conf_aware_target": target_qps,
})
preds_df.to_csv(f"{OUTPUT_DIR}/predictions.csv", index=False)

# Save scaling simulation (all 4 systems)
scaling_df = pd.DataFrame({
    "sample_index": np.arange(n_test),
    "actual_qps": Y_test,
    "hpa_replicas": hpa_replicas,
    "pahpa_replicas": pahpa_replicas,
    "conf_replicas": conf_replicas,
    "stab_replicas": stab_replicas,
    "hpa_latency_ms": hpa_latency * 1000,
    "pahpa_latency_ms": pahpa_latency * 1000,
    "conf_latency_ms": conf_latency * 1000,
    "stab_latency_ms": stab_latency * 1000,
})
scaling_df.to_csv(f"{OUTPUT_DIR}/scaling_simulation.csv", index=False)

# Save metrics
pd.DataFrame([
    {"system": "pahpa_point", **metrics_pahpa},
    {"system": "quantile_p50", **metrics_p50},
    {"system": "conf_aware", **metrics_conf},
]).to_csv(f"{OUTPUT_DIR}/prediction_metrics.csv", index=False)

# Save models
joblib.dump({
    "model": model_pahpa,
    "window": WINDOW_SIZE,
    "steps": 1,
    "features": "enhanced",
    "target_transform": TARGET_TRANSFORM,
    "metrics": metrics_pahpa,
}, f"{OUTPUT_DIR}/qps_point_model.joblib")

joblib.dump({
    "models": {str(q): quantile_models[q] for q in QUANTILES},
    "quantiles": QUANTILES,
    "window": WINDOW_SIZE,
    "steps": 1,
    "features": "enhanced",
    "target_transform": TARGET_TRANSFORM,
    "confidence_high": CONFIDENCE_HIGH,
    "confidence_low": CONFIDENCE_LOW,
    "metrics_p50": metrics_p50,
}, f"{OUTPUT_DIR}/qps_quantile_models.joblib")

print(f"Saved predictions, scaling simulation, metrics, and models to {OUTPUT_DIR}/")

In [ ]:
# Save training summary
summary = {
    "architecture": "LightGBM enhanced features + quantile regression + stability filter",
    "dataset_rows": int(len(series)),
    "window_size": WINDOW_SIZE,
    "target_transform": TARGET_TRANSFORM,
    "test_size": TEST_SIZE,
    "seed": SEED,
    "paper_reference": {
        "mape": 24.51, "mae": 7.59, "rmse": 10.53, "r2": 0.81
    },
    "pahpa_metrics": metrics_pahpa,
    "novelty1_p50_metrics": metrics_p50,
    "novelty1_conf_aware_metrics": metrics_conf,
    "prediction_interval": {
        "coverage_pct": round(coverage, 2),
        "avg_width": round(avg_width, 2),
        "median_width": round(median_width, 2),
    },
    "confidence_distribution": {
        "mean": round(float(np.mean(confidence)), 3),
        "high_pct": round(float(np.mean(confidence > CONFIDENCE_HIGH) * 100), 1),
        "medium_pct": round(float(np.mean((confidence > CONFIDENCE_LOW) & (confidence <= CONFIDENCE_HIGH)) * 100), 1),
        "low_pct": round(float(np.mean(confidence <= CONFIDENCE_LOW) * 100), 1),
    },
    "stability_filter": {
        "scale_up_consecutive": SCALE_UP_K,
        "scale_down_m_of_n": f"{SCALE_DOWN_M}/{SCALE_DOWN_N}",
        "burst_multiplier": BURST_MULTIPLIER,
    },
    "scaling_simulation": {
        "hpa_baseline": metrics_hpa_scale,
        "pahpa_point": metrics_pahpa_scale,
        "confidence_aware": metrics_conf_scale,
        "stability_aware": metrics_stab_scale,
    },
}

with open(f"{OUTPUT_DIR}/training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/final_output", "zip", OUTPUT_DIR)
print(f"Created: /kaggle/working/final_output.zip")
!ls -lh /kaggle/working/final_output.zip

## Summary

### Architecture

| Component | PAHPA Baseline | Novelty 1 | Novelty 1+2 (Full System) |
|---|---|---|---|
| Prediction | Point (single LightGBM) | Quantile (P10, P50, P90) | Quantile (P10, P50, P90) |
| Output | Single QPS value | Interval + confidence | Interval + confidence |
| Scaling decision | Trust prediction blindly | Risk-adjusted by confidence | Risk-adjusted + stability filtered |
| Stability filter | None | None | K-consecutive up, M/N down, burst bypass |
| QoS model | M/M/c queueing | M/M/c queueing | M/M/c queueing |

### Files to download from `output/`

| File | Where to put locally |
|---|---|
| `qps_point_model.joblib` | `prediction/models/` |
| `qps_quantile_models.joblib` | `prediction/models/` |
| `predictions.csv` | `data/` |
| `scaling_simulation.csv` | `data/` |
| `prediction_metrics.csv` | `data/` |
| `training_summary.json` | `training/reports/` |
| `actual_vs_predicted.png` | `training/reports/` |
| `scaling_analysis.png` | `training/reports/` |
| `qps_overview.png` | `training/reports/` |